# 01 — Bronze Layer: Raw Ingestion

**Purpose:** Download NYC TLC Yellow Taxi Parquet files and write them to the Bronze Delta table with zero transformation.

**Medallion layer:** 🥉 Bronze — raw, append-only, faithful copy of source

**What you will learn in this notebook:**
- How to download open data from a public URL using Python
- How to read Parquet files with PySpark
- How to validate schema before writing
- How to write a partitioned Delta Lake table
- How to add audit columns for traceability

---
**Dataset:** NYC TLC Yellow Taxi Trip Records  
**Source:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
**Volume:** ~3 million rows per month  

## 0. Setup — imports and config

In [ ]:
import os
import sys
import requests
from datetime import datetime

# Add project root to path so we can import src modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
# Change YEAR and MONTH to ingest a different month
YEAR  = 2024
MONTH = 1

# Data paths (relative to project root)
RAW_DIR     = os.path.join(PROJECT_ROOT, 'data', 'raw')
BRONZE_PATH = os.path.join(PROJECT_ROOT, 'data', 'bronze')

# TLC download URL pattern
TLC_BASE = "https://d37ci6vzurychx.cloudfront.net/trip-data"
FILE_URL = f"{TLC_BASE}/yellow_tripdata_{YEAR}-{MONTH:02d}.parquet"

print(f"Target file : {FILE_URL}")
print(f"Bronze path : {BRONZE_PATH}")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_PATH, exist_ok=True)

## 1. Start Spark Session

On **Databricks** — `SparkSession.builder.getOrCreate()` returns the already-running session.  
**Locally** — it creates a new local session with Delta Lake support.

In [ ]:
spark = (
    SparkSession.builder
    .appName("nyc-taxi-bronze")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"Spark UI      : http://localhost:4040  (if running locally)")

## 2. Download Source File

The NYC TLC publishes monthly Parquet files on a public CDN.  
We stream the download to handle large files without memory issues.  
Already-downloaded files are skipped (cached in `data/raw/`).

In [ ]:
filename  = FILE_URL.split("/")[-1]
dest_path = os.path.join(RAW_DIR, filename)

if os.path.exists(dest_path):
    size_mb = os.path.getsize(dest_path) / 1_048_576
    print(f"✓ Already downloaded: {filename} ({size_mb:.1f} MB) — skipping")
else:
    print(f"Downloading {FILE_URL} ...")
    response = requests.get(FILE_URL, stream=True, timeout=120)
    response.raise_for_status()

    total     = int(response.headers.get("content-length", 0))
    downloaded = 0

    with open(dest_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\r  {downloaded/total*100:.1f}%", end="", flush=True)

    size_mb = downloaded / 1_048_576
    print(f"\n✓ Download complete: {filename} ({size_mb:.1f} MB)")

## 3. Read and Explore the Raw Data

Before writing to Bronze, always explore the raw data first.  
This is good engineering practice — understand what you're working with.

In [ ]:
df_raw = spark.read.parquet(dest_path)

row_count = df_raw.count()
col_count = len(df_raw.columns)

print(f"Rows    : {row_count:,}")
print(f"Columns : {col_count}")
print(f"\nSchema:")
df_raw.printSchema()

In [ ]:
# Preview first 5 rows
# Note: display() works on Databricks; use show() locally
df_raw.show(5, truncate=False)

In [ ]:
# Basic descriptive statistics — spot anomalies before writing
df_raw.select(
    "fare_amount", "trip_distance", "passenger_count",
    "total_amount", "tip_amount"
).describe().show()

In [ ]:
# Check for nulls in critical columns
critical_cols = [
    "fare_amount", "tpep_pickup_datetime",
    "trip_distance", "passenger_count"
]

print("Null counts on critical columns:")
df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in critical_cols
]).show()

In [ ]:
# Payment type distribution — understand the data
print("Payment type breakdown:")
df_raw.groupBy("payment_type") \
      .count() \
      .orderBy("count", ascending=False) \
      .show()

In [ ]:
# Date range check — confirm data is for the expected month
df_raw.select(
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup")
).show(truncate=False)

## 4. Schema Validation

**Why validate before writing to Bronze?**  
TLC occasionally changes their schema. Catching a missing column at ingestion
is far better than discovering it when a downstream Silver job fails silently.

In [ ]:
REQUIRED_COLUMNS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID",
    "store_and_fwd_flag", "PULocationID", "DOLocationID",
    "payment_type", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge",
]

actual_cols = set(df_raw.columns)
missing     = [c for c in REQUIRED_COLUMNS if c not in actual_cols]
extra       = [c for c in actual_cols if c not in REQUIRED_COLUMNS]

if missing:
    raise ValueError(f"SCHEMA MISMATCH — missing columns: {missing}")

print(f"✓ All {len(REQUIRED_COLUMNS)} required columns present")
if extra:
    print(f"  Extra columns (OK, will be included): {extra}")

## 5. Add Audit Columns and Write to Bronze

Bronze = raw data + two audit columns:  
- `ingested_at` — when this row was loaded (for debugging)  
- `source_file` — which file it came from (for lineage)  

We **never** transform or filter Bronze data — it must be a faithful copy.

In [ ]:
df_bronze = (
    df_raw
    .withColumn("ingested_at",    F.current_timestamp())
    .withColumn("source_file",    F.lit(FILE_URL))
    .withColumn("pipeline_year",  F.lit(YEAR))
    .withColumn("pipeline_month", F.lit(MONTH))
)

print(f"Columns after audit fields added: {len(df_bronze.columns)}")
print(f"New columns: ingested_at, source_file, pipeline_year, pipeline_month")

In [ ]:
print(f"Writing {row_count:,} rows to Bronze Delta table...")
print(f"Path: {BRONZE_PATH}")

(
    df_bronze.write
    .format("delta")
    .mode("append")                              # append — never overwrite Bronze
    .partitionBy("pipeline_year", "pipeline_month")  # efficient partition pruning
    .save(BRONZE_PATH)
)

print(f"\n✓ Bronze write complete")

## 6. Verify the Bronze Table

In [ ]:
# Read back from Delta and verify row count matches
df_verify = spark.read.format("delta").load(BRONZE_PATH)

bronze_count = df_verify.count()
print(f"Rows in Bronze : {bronze_count:,}")
print(f"Original rows  : {row_count:,}")
print(f"Match          : {'✓ YES' if bronze_count == row_count else '✗ NO — investigate!'}")

In [ ]:
# Inspect Delta table history — Delta Lake keeps a transaction log
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, BRONZE_PATH)
delta_table.history().select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [ ]:
# Confirm partitions were created correctly
df_verify.groupBy("pipeline_year", "pipeline_month") \
         .count() \
         .orderBy("pipeline_year", "pipeline_month") \
         .show()

## 7. Bronze Data Quality Check

Quick inline checks before handing off to Silver.  
Full DQ module is in `data_quality/checks.py`.

In [ ]:
df_b = spark.read.format("delta").load(BRONZE_PATH)
total = df_b.count()

fare_nulls = df_b.filter(F.col("fare_amount").isNull()).count()
null_rate  = fare_nulls / total if total > 0 else 1.0

checks = [
    ("Row count >= 100,000",       total >= 100_000,          f"{total:,}"),
    ("fare_amount null rate < 1%", null_rate < 0.01,          f"{null_rate*100:.3f}%"),
    ("ingested_at column exists",  "ingested_at" in df_b.columns, str("ingested_at" in df_b.columns)),
    ("source_file column exists",  "source_file" in df_b.columns, str("source_file" in df_b.columns)),
]

print("Bronze Data Quality Report")
print("-" * 55)
all_passed = True
for name, passed, value in checks:
    icon = "✓" if passed else "✗"
    print(f"  {icon}  {name:<35} {value}")
    if not passed:
        all_passed = False
print("-" * 55)
print(f"  {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED — review before Silver'}")

## ✅ Summary

| Step | What happened |
|------|---------------|
| Download | NYC TLC Parquet file fetched from CDN |
| Explore | Schema, nulls, date range, distributions reviewed |
| Validate | All 18 required columns confirmed present |
| Audit | `ingested_at` and `source_file` columns added |
| Write | Appended to Bronze Delta table, partitioned by year/month |
| Verify | Row count confirmed, Delta history inspected |
| DQ | All Bronze quality checks passed |

**Next:** Open `02_silver_cleaning.ipynb` to clean and validate this data.